# 11 - PySpark: Parquet e Modelo Medalhao

Bronze > Silver > Gold com PySpark e Parquet.

## 0. Setup

In [ ]:
import os, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master('local[*]').appName('datalab-medalhao').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

BASE = '/tmp/datalab_medalhao'
if os.path.exists(BASE):
    shutil.rmtree(BASE)
os.makedirs(BASE)
print(f'Base: {BASE}')

## 1. Bronze - Dados crus (com duplicatas e nulos)

In [ ]:
bronze_path = os.path.join(BASE, 'bronze')

dados = [
    (1, 'Ana', 'TI', 9500.0),
    (2, 'Joao', 'RH', 7200.0),
    (3, 'Maria', 'TI', 11000.0),
    (2, 'Joao', 'RH', 7200.0),   # duplicata
    (4, None, 'Financeiro', 8800.0),  # nome nulo
    (5, 'Lucia', 'RH', None),   # salario nulo
    (6, 'Carlos', 'Vendas', 10500.0),
    (None, None, None, None),   # linha toda nula
]

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('nome', StringType(), True),
    StructField('depto', StringType(), True),
    StructField('salario', DoubleType(), True),
])

df_bronze = spark.createDataFrame(dados, schema=schema)
df_bronze.write.mode('overwrite').parquet(bronze_path)
print(f'Bronze registros: {df_bronze.count()}')
df_bronze.show()

## 2. Silver - Limpeza e validacao

In [ ]:
silver_path = os.path.join(BASE, 'silver')

df_silver = (
    spark.read.parquet(bronze_path)
    .dropDuplicates()
    .filter(F.col('id').isNotNull())
    .filter(F.col('nome').isNotNull())
    .filter(F.col('salario').isNotNull())
    .withColumn('salario', F.col('salario').cast(DoubleType()))
)

df_silver.write.mode('overwrite').parquet(silver_path)
print(f'Silver registros: {df_silver.count()}')
df_silver.show()

## 3. Gold - Agregacao dimensional

In [ ]:
gold_path = os.path.join(BASE, 'gold')

df_gold = (
    df_silver
    .groupBy('depto')
    .agg(
        F.count('*').alias('qtd_func'),
        F.round(F.avg('salario'), 2).alias('salario_medio'),
        F.max('salario').alias('salario_max'),
    )
)

df_gold.write.mode('overwrite').parquet(gold_path)
print('Gold:')
df_gold.show()

## 4. Validacao

In [ ]:
df_check = spark.read.parquet(gold_path)
assert df_check.filter(F.col('depto').isNull()).count() == 0, 'Gold contem nulos!'
assert df_check.filter(F.col('qtd_func').isNull()).count() == 0, 'qtd_func nulo!'
print('Validacao OK: sem nulos no Gold')

## 5. Cleanup

In [ ]:
shutil.rmtree(BASE)
print(f'Diretorio {BASE} removido')

## Conclusao

Bronze = bruto, Silver = limpo/validado, Gold = agregado/dimensional.

In [ ]:
spark.stop()